# Xarray-spatial
### User Guide: Least-Cost Corridor Analysis
-----

The `least_cost_corridor` function identifies zones of low cumulative travel cost between two (or more) source locations on a friction surface. Unlike a single-cell path (e.g., A* search), a corridor shows **all cells** within a cost threshold of the optimal route.

Typical use cases:
- **Wildlife connectivity**: connecting two habitat patches through the cheapest terrain
- **Infrastructure routing**: finding broad zones suitable for roads or pipelines
- **Conservation planning**: prioritizing land parcels that lie along low-cost connections

**How it works:**
1. Compute `cost_distance` from source A and from source B.
2. Sum the two surfaces: `corridor = cd_A + cd_B`.
3. Normalize by subtracting the minimum: cells on the optimal route get value 0.
4. Optionally threshold to produce a binary corridor mask.

**Contents:**
- [Setup](#Setup)
- [1. Basic corridor between two points](#1.-Basic-corridor-between-two-points)
- [2. Corridor with variable friction](#2.-Corridor-with-variable-friction)
- [3. Thresholding: absolute vs relative](#3.-Thresholding:-absolute-vs-relative)
- [4. Barriers force corridor detours](#4.-Barriers-force-corridor-detours)
- [5. Pre-computed cost-distance surfaces](#5.-Pre-computed-cost-distance-surfaces)
- [6. Multi-source pairwise corridors](#6.-Multi-source-pairwise-corridors)

-----

## Setup

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import least_cost_corridor, cost_distance

In [ ]:
def make_raster(data, res=1.0):
    """Helper: create a DataArray with y/x coordinates."""
    h, w = data.shape
    raster = xr.DataArray(
        data.astype(np.float64),
        dims=['y', 'x'],
        attrs={'res': (res, res)},
    )
    raster['y'] = np.arange(h) * res
    raster['x'] = np.arange(w) * res
    return raster


def plot_corridor(arrays, titles, cmaps=None, figsize=None):
    """Plot multiple arrays side by side."""
    n = len(arrays)
    if figsize is None:
        figsize = (5 * n, 4)
    if cmaps is None:
        cmaps = ['viridis'] * n
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, arr, title, cmap in zip(axes, arrays, titles, cmaps):
        data = arr.values if hasattr(arr, 'values') else arr
        im = ax.imshow(data, cmap=cmap, origin='upper')
        ax.set_title(title)
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

## 1. Basic corridor between two points

With uniform friction, the corridor cost surface is symmetric and centred on the straight-line path between the two sources.

In [ ]:
n = 31
friction_data = np.ones((n, n))

# Two sources on opposite sides
src_a_data = np.zeros((n, n))
src_a_data[15, 3] = 1.0

src_b_data = np.zeros((n, n))
src_b_data[15, 27] = 1.0

friction = make_raster(friction_data)
src_a = make_raster(src_a_data)
src_b = make_raster(src_b_data)

corridor = least_cost_corridor(friction, src_a, src_b)

plot_corridor(
    [corridor],
    ['Corridor (uniform friction)'],
    cmaps=['magma'],
    figsize=(8, 5),
)

print(f"Min corridor cost (optimal route): {float(corridor.min()):.4f}")
print(f"Max corridor cost (corners):       {float(corridor.max()):.4f}")

The darkest band shows cells on or near the optimal route (cost = 0). Brighter values are farther from the optimal connection, measured in accumulated cost.

## 2. Corridor with variable friction

When friction varies across the landscape, the corridor bends to follow cheaper terrain. Here we add a high-friction band and a low-friction channel that the corridor routes through.

In [ ]:
n = 31
friction_var = np.ones((n, n)) * 2.0

# High-cost zone across the middle
friction_var[10:21, :] = 8.0

# Low-cost channel through the high-cost zone
friction_var[14:17, :] = 1.0

src_a_data = np.zeros((n, n))
src_a_data[15, 2] = 1.0

src_b_data = np.zeros((n, n))
src_b_data[15, 28] = 1.0

friction_v = make_raster(friction_var)
sa = make_raster(src_a_data)
sb = make_raster(src_b_data)

corridor_var = least_cost_corridor(friction_v, sa, sb)

plot_corridor(
    [friction_v, corridor_var],
    ['Friction surface', 'Corridor'],
    cmaps=['YlOrRd', 'magma'],
    figsize=(12, 5),
)

The corridor is concentrated along the low-cost channel. Cells outside the channel have much higher corridor cost because any route through them must also cross the high-friction zone.

## 3. Thresholding: absolute vs relative

The `threshold` parameter masks out cells that deviate too far from the optimal route. This turns the continuous corridor surface into a discrete corridor zone.

- **Absolute** (`relative=False`): cells with normalized cost > threshold are masked.
- **Relative** (`relative=True`): threshold is a fraction of the minimum corridor cost. For example, `threshold=0.10` keeps cells within 10% of the optimal cost.

In [ ]:
# Reuse uniform friction setup
n = 31
friction_u = make_raster(np.ones((n, n)))

sa_data = np.zeros((n, n))
sa_data[15, 3] = 1.0

sb_data = np.zeros((n, n))
sb_data[15, 27] = 1.0

sa_u = make_raster(sa_data)
sb_u = make_raster(sb_data)

# Absolute threshold
corridor_abs = least_cost_corridor(friction_u, sa_u, sb_u, threshold=3.0)

# Relative threshold (10% of minimum corridor cost)
corridor_rel = least_cost_corridor(
    friction_u, sa_u, sb_u, threshold=0.10, relative=True
)

plot_corridor(
    [corridor_abs, corridor_rel],
    ['Absolute threshold=3.0', 'Relative threshold=10%'],
    cmaps=['magma', 'magma'],
    figsize=(12, 5),
)

# Count cells in each corridor
print(f"Cells in absolute corridor: {int(np.sum(np.isfinite(corridor_abs.values)))}")
print(f"Cells in relative corridor: {int(np.sum(np.isfinite(corridor_rel.values)))}")

The relative threshold adapts to the scale of the corridor cost. For long-distance corridors with high total cost, a 10% relative threshold produces a wider corridor than for short-distance ones. The absolute threshold gives a fixed-width band in cost units regardless of distance.

## 4. Barriers force corridor detours

NaN cells in the friction surface are impassable. The corridor routes around them, just as `cost_distance` does.

In [ ]:
n = 31
friction_barrier = np.ones((n, n))

# Vertical wall with a gap
friction_barrier[5:26, 15] = np.nan
friction_barrier[15, 15] = 1.0  # gap at the midpoint

sa_data = np.zeros((n, n))
sa_data[15, 3] = 1.0

sb_data = np.zeros((n, n))
sb_data[15, 27] = 1.0

fric_b = make_raster(friction_barrier)
sa_b = make_raster(sa_data)
sb_b = make_raster(sb_data)

corridor_barrier = least_cost_corridor(fric_b, sa_b, sb_b)

# Show friction with barrier visible
barrier_vis = friction_barrier.copy()
barrier_vis[np.isnan(barrier_vis)] = 0

plot_corridor(
    [make_raster(barrier_vis), corridor_barrier],
    ['Friction (dark = barrier)', 'Corridor (routes through gap)'],
    cmaps=['gray', 'magma'],
    figsize=(12, 5),
)

All routes between the two sources funnel through the gap in the wall. The corridor cost surface reflects this bottleneck -- cells near the gap have low corridor cost, while cells far from it have high cost because detour paths through the gap are longer.

## 5. Pre-computed cost-distance surfaces

If you have already computed cost-distance surfaces (e.g., to reuse them for multiple corridor pairs), pass them directly with `precomputed=True` to skip the internal `cost_distance` calls.

In [ ]:
n = 21
friction_p = make_raster(np.ones((n, n)))

sa_data = np.zeros((n, n))
sa_data[10, 2] = 1.0
sb_data = np.zeros((n, n))
sb_data[10, 18] = 1.0

sa_p = make_raster(sa_data)
sb_p = make_raster(sb_data)

# Compute cost-distance surfaces once
cd_a = cost_distance(sa_p, friction_p)
cd_b = cost_distance(sb_p, friction_p)

# Reuse for corridor (skips redundant cost_distance calls)
corridor_pre = least_cost_corridor(
    friction_p, cd_a, cd_b, precomputed=True
)

# Compare with the regular call
corridor_reg = least_cost_corridor(friction_p, sa_p, sb_p)

diff = np.abs(corridor_pre.values - corridor_reg.values)
print(f"Max difference: {np.nanmax(diff):.10f} (should be ~0)")

## 6. Multi-source pairwise corridors

With three or more source locations, `least_cost_corridor` can compute corridors for every pair at once. Pass the sources as a list with `pairwise=True`. The result is an `xr.Dataset` with one variable per pair.

In [ ]:
n = 31
friction_m = make_raster(np.ones((n, n)))

# Three habitat patches
positions = [(5, 5), (5, 25), (25, 15)]
sources = []
for r, c in positions:
    s = np.zeros((n, n))
    s[r, c] = 1.0
    sources.append(make_raster(s))

corridors = least_cost_corridor(
    friction_m, sources=sources, pairwise=True
)

print("Corridor pairs:", list(corridors.data_vars))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, corridors.data_vars):
    im = ax.imshow(corridors[name].values, cmap='magma', origin='upper')
    ax.set_title(name)
    # Mark source positions
    for r, c in positions:
        ax.plot(c, r, 'w*', markersize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

Each subplot shows the corridor between one pair of sources (marked with white stars). With uniform friction, corridors follow straight-line paths. With variable friction, each pair would route through different low-cost terrain.

## Summary

| Parameter | Effect |
|---|---|
| `friction` | Cost surface (NaN = barrier) |
| `source_a`, `source_b` | Two source rasters |
| `sources` + `pairwise=True` | Compute corridors for all pairs |
| `threshold` | Mask cells beyond this cost deviation |
| `relative=True` | Interpret threshold as a fraction of minimum corridor cost |
| `precomputed=True` | Skip internal cost_distance calls, use pre-computed surfaces |

**When to use corridor analysis vs pathfinding:**
- Use `a_star_search` when you need a single optimal route (one cell wide).
- Use `least_cost_corridor` when you need a **zone** of low-cost connectivity between two regions.

### References
- ArcGIS Corridor: https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/corridor.htm
- Beier, P., Majka, D. R., & Spencer, W. D. (2008). Forks in the Road: Choices in Procedures for Designing Wildland Linkages. *Conservation Biology*, 22(4), 836-851.